In [1]:
%pip install evaluate
%pip install sacrebleu

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Les importations
import evaluate
import matplotlib.pyplot as plt
import numpy as np
#import tensorflow as tf
import torch
from datasets import Dataset, DatasetDict, load_dataset
#from comet import download_model, load_from_checkpoint
from torch.cuda.amp import GradScaler, autocast
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    get_linear_schedule_with_warmup,
)

SOURCE_LANG_CODE_NLLB =  "fra_Latn" 
TARGET_LANG_CODE_NLLB =  "arb_Latn" 
max_input_length = 128
max_target_length = 128


class MemoryCleanupCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()  # Libérer la mémoire GPU inutilisée

class LossTrackerCallback(TrainerCallback):
    def __init__(self):
        self.history = {
            "step": [],
            "train_loss": [],
            "eval_loss": []
        }

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        
        if "loss" in logs:
            self.history["step"].append(state.global_step)
            self.history["train_loss"].append(logs["loss"])
            self.history["eval_loss"].append(None)

        if "eval_loss" in logs:
            self.history["step"].append(state.global_step)
            self.history["train_loss"].append(None)
            self.history["eval_loss"].append(logs["eval_loss"])


def training (modele_name, train_test_val_dataset) : 
    
    loss_tracker = LossTrackerCallback()
    model = AutoModelForSeq2SeqLM.from_pretrained(modele_name)
    tokenizer = AutoTokenizer.from_pretrained(modele_name)

    def postprocess_text(preds, labels):
        preds = [pred.strip() for pred in preds]
        labels = [[label.strip()] for label in labels]  # ROUGE/METEOR attend des listes imbriquées
        return preds, labels

    def compute_metrics(eval_preds):
        preds, labels = eval_preds

        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        
        if isinstance(preds, tuple):
            preds = preds[0]

        # Décodage des prédictions et labels
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        #labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        # Nettoyage du texte
        decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

        # Charger les métriques
        bleu_metric = evaluate.load("sacrebleu")
        rouge_metric = evaluate.load("rouge")
        meteor_metric = evaluate.load("meteor")

        # 🏆 Calcul des métriques
        bleu_result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
        rouge_result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
        meteor_result = meteor_metric.compute(predictions=decoded_preds, references=decoded_labels)

        # 📊 Récupération des scores
        result = {
            "bleu": bleu_result["score"],
            "rouge": rouge_result["rougeL"],  # Utilisation du score ROUGE-L
            "meteor": meteor_result["meteor"],
        }

        # Arrondir les scores
        result = {k: round(v, 4) for k, v in result.items()}
        return result


    def extract_langage(examples) :
        # On va extraire le langage arabe et le langage français
        inputs = [fr['fr'] for fr in examples["translation"]]
        target = [arb['arb'] for arb in examples["translation"]]
        return {'inputs' : inputs, 'target' : target}



    def preprocess_function(examples):
        tokenizer.src_lang = SOURCE_LANG_CODE_NLLB
        model_inputs = tokenizer(
            examples["inputs"],
            max_length=max_input_length,
            truncation=True
        )

        # Nettoyer les cibles : enlever None ou chaînes vides
        clean_targets = []
        for t in examples["target"]:
            if isinstance(t, str) and t.strip():
                clean_targets.append(t)
            else:
                clean_targets.append(" ")

        tokenizer.tgt_lang = TARGET_LANG_CODE_NLLB
        labels = tokenizer(
            clean_targets,
            max_length=max_target_length,
            truncation=True,
            text_target=clean_targets  # Pour compatibilité future
        )

        model_inputs["labels"] = labels["input_ids"]
        
        return model_inputs

    
    def train_and_valid_loss() :

        loss_dict = loss_tracker.history
        steps = loss_dict["step"]

        train_loss = [
            l for l in loss_dict["train_loss"] if l is not None]
        train_steps = [
            s for s, l in zip(steps, loss_dict["train_loss"]) if l is not None]

        eval_loss = [
            l for l in loss_dict["eval_loss"] if l is not None
        ]
        eval_steps = [
            s for s, l in zip(steps, loss_dict["eval_loss"]) if l is not None
        ]

        plt.plot(train_steps, train_loss, label="Train Loss")
        plt.plot(eval_steps, eval_loss, label="Validation Loss")
        plt.xlabel("Steps")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.show()


    # On va appliquer la fonction sur notre dataset
    train_dataset = train_test_val_dataset.map(extract_langage, batched=True, remove_columns=["translation"])
    # On va appliquer la fonction sur notre dataset
    train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=["inputs", "target"])
    # Dataloader
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)


    early =  EarlyStoppingCallback(early_stopping_patience = 5)
    #loss_tracker = LossTrackerCallback()


    training_args = Seq2SeqTrainingArguments(
        output_dir=f"../src/{modele_name}",
        report_to="none",
        #hub_model_id= os.environ.get('HUB_ID_2'),  # ✅ Ajoute ton Repo ID
        eval_strategy="steps",
        eval_steps=100,              
        logging_steps=100, 
        learning_rate=2e-5,
        #learning_rate=1e-5,
        gradient_checkpointing=True,  # ✅ Active la réduction mémoire
        #use_cache = False,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=50,
        metric_for_best_model="eval_loss",
        predict_with_generate=True,
        fp16=True, 
        #push_to_hub=True,
        remove_unused_columns=False,
        load_best_model_at_end=True,
        save_strategy="steps",
        #save_steps = 50,
        #resume_from_checkpoint=True,
        gradient_accumulation_steps=2,

    )




    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset["train"],
        eval_dataset=train_dataset["val"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[early,loss_tracker, MemoryCleanupCallback], #CheckpointCleanupCallback()
    )
    torch.cuda.empty_cache()  # Libérer la mémoire GPU inutilisée

    trainer.train()

    results = trainer.evaluate(train_dataset["test"]) 

    return trainer,results




In [3]:
def dataset_loader(path) :
    with open(path, "r") as file:
        data = file.readlines()

    data_clean = [line for line in data if line.strip()]

    return data_clean


In [4]:
PATH = "/kaggle/input/arb-fr/"

data_arb = dataset_loader(PATH + "for_dataset_arb.txt")
data_fr = dataset_loader(PATH + "for_dataset_fr.txt")

In [5]:
# # for trying 
# data_arb = data_arb[:100] 
# data_fr = data_fr[:100]

In [6]:
# Création du dictionnaire
dataset_dict = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb, data_fr)]}

# Création du dataset
dataset = Dataset.from_dict(dataset_dict)
# On va diviser notre dataset en deux parties
 #train_dataset =  DatasetDict({"train":test['train']})

train_test_dataset = dataset.train_test_split(test_size=0.2, seed= 42, shuffle= True)

test_val_dataset = train_test_dataset['test'].train_test_split(test_size= 0.5, seed= 42)

train_test_val_dataset = DatasetDict ({
    'train' : train_test_dataset['train'],
    'val'   : test_val_dataset['train'],
    'test'  : test_val_dataset['test']
})
train_test_val_dataset.save_to_disk("../Data/train_test_split")


Saving the dataset (0/1 shards):   0%|          | 0/10402 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1301 [00:00<?, ? examples/s]

In [7]:
import os
# À placer impérativement avant l'import de torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# Les after....

In [8]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)


In [9]:
#from src.models import training
from datasets import load_from_disk
from transformers import get_linear_schedule_with_warmup, AutoTokenizer,TrainerCallback,AutoModelForCausalLM, DataCollatorForSeq2Seq,AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer,EarlyStoppingCallback

### loading le train,test, val et model et tokenizer

In [10]:
train_test_val_dataset = load_from_disk("../Data/train_test_split")

### loading the model et tokenizer

In [11]:
nllb = "facebook/nllb-200-distilled-600M"
# model = AutoModelForSeq2SeqLM.from_pretrained(modele_name)
# tokenizer = AutoTokenizer.from_pretrained(modele_name)

In [12]:
trainer_nllb, nllb_evaluation = training(nllb, train_test_val_dataset)

nllb_evaluation

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.09 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.83 GiB is free. Including non-PyTorch memory, this process has 12.73 GiB memory in use. Of the allocated memory 12.42 GiB is allocated by PyTorch, and 109.46 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
#trainer_nllb.evaluate(train_test_val_dataset["test"])

## ByT5

In [ ]:
byt5 = "google/byt5-small"

trainer_byt5, evaluation_byt5 = training(byt5, train_test_val_dataset)


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

...

Step,Training Loss,Validation Loss,Bleu,Rouge,Meteor
10,25.229556,2.959781,0.044400,0.007900,0.008200
20,22.599506,2.774974,0.043200,0.017400,0.008300
30,19.722916,2.638760,0.035400,0.026000,0.007000
40,19.646062,2.487436,0.034900,0.021900,0.005500
50,18.869833,2.325507,0.041200,0.021900,0.005500
60,16.843263,2.214199,0.048900,0.025300,0.005500
70,17.364859,2.141039,0.044200,0.019700,0.003500
80,16.795639,2.072257,0.028100,0.030200,0.006200
90,15.432336,2.020202,0.029700,0.020000,0.003500
100,16.088008,1.980718,0.026600,0.023700,0.003500


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=150, training_loss=17.61640146891276, metrics={'train_runtime': 703.1272, 'train_samples_per_second': 11.378, 'train_steps_per_second': 0.213, 'total_flos': 1837499154432000.0, 'train_loss': 17.61640146891276, 'epoch': 50.0})

### Afri-byt5

In [ ]:
model_afri = "masakhane/afri-byt5-base"

trainer_afri = training(model_afri, train_test_val_dataset)
trainer_afri.train()